# TP3 — De la probabilité à la décision

## Mission
Au moment où une commande vient d’être validée et où une date estimée de livraison a été communiquée au client, peut-on **identifier les commandes présentant un risque de retard** ?

Dans les TP précédents, nous prédisions une quantité. Ici :

$$
X \rightarrow P(Y=1 \mid X) \rightarrow \text{décision}
$$

avec `is_late = 1` pour une commande livrée en retard.

## Objectifs

- entraîner une régression logistique
- passer d'une probabilité à une décision
- étudier le seuil
- interpréter la matrice de confusion et les métriques de classification
- comprendre le déséquilibre des classes
- expérimenter le sampling et analyser de manière critique SMOTE.

## 1. Préparation

In [ ]:
!pip -q install imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay)
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42

In [ ]:
url = "https://raw.githubusercontent.com/ndiayemairame/miage-machine-learning/main/data/orders_ml.csv"
orders = pd.read_csv(url)
print("Dimensions :", orders.shape)
orders.head()

## 2. Comprendre la cible
Notre unité d'observation reste **une commande**. La cible est `is_late`.

In [ ]:
display(orders["is_late"].value_counts())

**Q1.** Quelle est la classe minoritaire ?

**Q2.** Quelle proportion des commandes appartient à cette classe ?

**Q3.** Si un modèle prédisait systématiquement `0`, quelle accuracy obtiendrait-il environ ?

**Q4.** Cette accuracy suffirait-elle pour considérer le modèle comme utile ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 3. Train, validation et test
Nous conservons trois jeux. La séparation est stratifiée afin de préserver approximativement la proportion de retards.

In [ ]:
train, temp = train_test_split(orders, test_size=0.30, random_state=RANDOM_STATE,
                               stratify=orders["is_late"])
validation, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_STATE,
                                    stratify=temp["is_late"])

for name, df in [("train", train), ("validation", validation), ("test", test)]:
    print(name, len(df), "observations —", round(100*df["is_late"].mean(), 2), "% de retards")

## 4. Informations disponibles au moment de la prédiction
L'instant de prédiction reste : **commande validée, date estimée déjà communiquée au client**.

In [ ]:
features = [
    "promised_delivery_days", "purchase_month", "purchase_day_of_week", "purchase_hour",
    "n_items", "n_products", "n_sellers", "total_price", "total_freight",
    "mean_product_weight_g", "max_product_weight_g",
    "mean_product_volume_cm3", "max_product_volume_cm3", "n_categories",
    "freight_ratio", "seller_customer_distance_mean", "seller_customer_distance_max",
    "same_state_share", "payment_installments_max"
]
target = "is_late"

X_train, y_train = train[features], train[target]
X_val, y_val = validation[features], validation[target]
X_test, y_test = test[features], test[target]

**Q5.** Pourquoi `delivery_time_days`, `carrier_time_days` et `review_score` ne doivent-elles pas être utilisées ici comme variables explicatives ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 5. Première régression logistique
Le modèle estime une probabilité :

$$
P(Y=1 \mid X)
$$

Ici : la probabilité estime qu'une commande soit livrée en retard.

In [ ]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

logistic_model = Pipeline([
    ("preprocessing", numeric_preprocessor),
    ("logistic", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
proba_val = logistic_model.predict_proba(X_val)[:, 1]
pd.Series(proba_val, name="probabilite_retard").describe()

**Q6.** Que représente une probabilité prédite de `0.20` ?

**Q7.** Cette probabilité permet-elle, à elle seule, de dire que la commande sera classée « retard » ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 6. De la probabilité à la décision
Commençons avec un seuil de `0.5`.

In [ ]:
def classification_metrics(y_true, y_pred):
    return pd.Series({
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    })

threshold = 0.5
pred_val = (proba_val >= threshold).astype(int)

classification_metrics(y_val, pred_val)

**Q8.** Comparez l'accuracy obtenue avec la baseline.

**Q9.** Combien de commandes en retard le modèle détecte-t-il ?

**Q10.** Pourquoi l'accuracy donne-t-elle ici une vision incomplète ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 7. Matrice de confusion

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val, pred_val, display_labels=["Pas retard", "Retard"], values_format="d")
plt.title("Matrice de confusion — seuil = 0.5")
plt.show()

**Q11.** Dans notre problème, que représente un faux négatif ?

**Q12.** Que représente un faux positif ?

**Q13.** Si l'objectif est de détecter davantage de vrais retards, quel type d'erreur cherche-t-on en priorité à réduire ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 8. Le seuil est une décision

In [ ]:
thresholds = [0.50, 0.30, 0.20, 0.10, 0.075, 0.05]
results = []
for t in thresholds:
    pred = (proba_val >= t).astype(int)
    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "n_alerts": pred.sum()
    })
threshold_results = pd.DataFrame(results)
threshold_results

In [ ]:
plt.figure(figsize=(8,5))
for metric in ["precision", "recall", "f1"]:
    plt.plot(threshold_results["threshold"], threshold_results[metric],
             marker="o", label=metric)
plt.xlabel("Seuil")
plt.ylabel("Score")
plt.title("Effet du seuil de décision")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**Q14.** Que devient le rappel lorsque le seuil diminue ?

**Q15.** Que devient généralement la précision ?

**Q16.** Pourquoi ne peut-on pas choisir un seuil uniquement en maximisant le rappel ?

**Q17.** Si l'entreprise peut traiter un nombre limité d'alertes, quelle information supplémentaire faut-il considérer ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 9. Évaluer sans fixer un seuil unique
Étudions ROC et Precision-Recall.

In [ ]:
print("ROC-AUC :", round(roc_auc_score(y_val, proba_val), 3))
print("Average Precision :", round(average_precision_score(y_val, proba_val), 3))

RocCurveDisplay.from_predictions(y_val, proba_val)
plt.title("Courbe ROC — validation")
plt.show()

PrecisionRecallDisplay.from_predictions(y_val, proba_val)
plt.title("Courbe Precision-Recall — validation")
plt.show()

**Q18.** Pourquoi ces courbes apportent-elles une information différente des métriques calculées au seuil `0.5` ?

**Q19.** Pourquoi Precision-Recall est-elle particulièrement intéressante lorsque la classe positive est rare ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

# 10. Agir sur le déséquilibre des classes

Nous modifions **uniquement le jeu d'entraînement**.

> **Validation et test ne sont jamais rééquilibrés.**

Ils doivent conserver la distribution réelle.

In [ ]:
X_train_prepared = numeric_preprocessor.fit_transform(X_train)
X_val_prepared = numeric_preprocessor.transform(X_val)

print("Avant rééchantillonnage :")
print(y_train.value_counts())

## 11. Random Oversampling et Random Undersampling
L'oversampling duplique des observations minoritaires, l'undersampling retire des observations majoritaires.

In [ ]:
samplers = {
    "Naturel": None,
    "Random oversampling": RandomOverSampler(random_state=RANDOM_STATE),
    "Random undersampling": RandomUnderSampler(random_state=RANDOM_STATE)
}
sampling_results = []

for name, sampler in samplers.items():
    if sampler is None:
        X_res, y_res = X_train_prepared, y_train.to_numpy()
    else:
        X_res, y_res = sampler.fit_resample(X_train_prepared, y_train)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_res, y_res)
    proba = model.predict_proba(X_val_prepared)[:,1]
    pred = (proba >= 0.5).astype(int)

    sampling_results.append({
        "méthode": name, "n_train": len(y_res),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, proba),
        "average_precision": average_precision_score(y_val, proba)
    })

pd.DataFrame(sampling_results)

**Q20.** Comment le comportement du classifieur change-t-il ?

**Q21.** Le rééquilibrage améliore-t-il toutes les métriques simultanément ?

**Q22.** Pourquoi mesurer les performances sur la validation non rééquilibrée ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

# 12. SMOTE : créer de nouvelles observations
SMOTE ne se contente pas de dupliquer des observations : il crée des observations synthétiques à partir d'observations minoritaires existantes.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_smote, y_smote = smote.fit_resample(X_train_prepared, y_train)

print("Taille avant SMOTE :", len(y_train))
print("Taille après SMOTE :", len(y_smote))
print(pd.Series(y_smote).value_counts())

smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_smote, y_smote)

proba_smote = smote_model.predict_proba(X_val_prepared)[:,1]
pred_smote = (proba_smote >= 0.5).astype(int)

display(classification_metrics(y_val, pred_smote))
print("ROC-AUC :", round(roc_auc_score(y_val, proba_smote), 3))
print("Average Precision :", round(average_precision_score(y_val, proba_smote), 3))

**Q23.** Comparez SMOTE aux méthodes précédentes. Observez-vous une amélioration systématique ?

**Q24.** Davantage d'observations d'entraînement signifie-t-il nécessairement davantage d'information réelle ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 13. Que fabrique réellement SMOTE ?
Pour visualiser le phénomène, construisons une expérience en deux dimensions. Cette représentation est volontairement simplifiée.

In [ ]:
smote_features = ["promised_delivery_days", "seller_customer_distance_mean"]

imp2 = SimpleImputer(strategy="median")
sc2 = StandardScaler()
X2 = sc2.fit_transform(imp2.fit_transform(train[smote_features]))
y2 = train[target].to_numpy()

X2_smote, y2_smote = SMOTE(random_state=RANDOM_STATE).fit_resample(X2, y2)
X2_synthetic = X2_smote[len(X2):]

print("Observations originales :", len(X2))
print("Observations synthétiques :", len(X2_synthetic))

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
maj = np.where(y2 == 0)[0]
mino = np.where(y2 == 1)[0]

maj_s = rng.choice(maj, min(2500, len(maj)), replace=False)
min_s = rng.choice(mino, min(1200, len(mino)), replace=False)
syn_s = rng.choice(len(X2_synthetic), min(2000, len(X2_synthetic)), replace=False)

plt.figure(figsize=(9,6))
plt.scatter(X2[maj_s,0], X2[maj_s,1], alpha=.20, s=12, label="Pas retard — réel")
plt.scatter(X2[min_s,0], X2[min_s,1], alpha=.45, s=18, label="Retard — réel")
plt.scatter(X2_synthetic[syn_s,0], X2_synthetic[syn_s,1], alpha=.45, s=18,
            marker="x", label="Retard — synthétique")
plt.xlabel("promised_delivery_days — standardisé")
plt.ylabel("seller_customer_distance_mean — standardisé")
plt.title("Observations réelles et synthétiques")
plt.legend()
plt.show()

**Q25.** Où se situent les observations synthétiques par rapport aux vrais retards ?

**Q26.** Certaines apparaissent-elles dans des régions également occupées par des commandes `pas retard` ?

**Q27.** Quelles conséquences cela pourrait-il avoir sur la frontière de décision ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 14. Quantifier l'ambiguïté locale
Dans l'espace numérique complet, comparons pour un échantillon de points synthétiques la distance au retard réel le plus proche et au `pas retard` réel le plus proche.

In [ ]:
X_original = X_train_prepared
y_original = y_train.to_numpy()

X_real_late = X_original[y_original == 1]
X_real_not_late = X_original[y_original == 0]
X_synthetic = X_smote[len(X_original):]

rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(X_synthetic), min(5000, len(X_synthetic)), replace=False)
X_syn_sample = X_synthetic[idx]

nn_late = NearestNeighbors(n_neighbors=1).fit(X_real_late)
nn_not = NearestNeighbors(n_neighbors=1).fit(X_real_not_late)

d_late = nn_late.kneighbors(X_syn_sample)[0].ravel()
d_not = nn_not.kneighbors(X_syn_sample)[0].ravel()

ambiguity = pd.DataFrame({
    "distance_retard": d_late,
    "distance_pas_retard": d_not
})
ambiguity.head()

In [ ]:
share = (ambiguity["distance_pas_retard"] < ambiguity["distance_retard"]).mean()
print("Points synthétiques plus proches d'un pas-retard réel que d'un retard réel :",
      round(100*share, 2), "%")

**Q28.** Comment interprétez-vous ce résultat ?

**Q29.** Pourquoi cette expérience empêche-t-elle de considérer SMOTE comme une solution automatique au déséquilibre ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 15. Et les variables catégorielles ?

Après One-Hot Encoding, deux paiements peuvent être représentés ainsi :

```text
credit_card    boleto
    1             0
    0             1
```

Une interpolation numérique naïve pourrait produire :

```text
credit_card    boleto
   0.4           0.6
```

**Q30.** Cette observation correspond-elle directement à une modalité réelle de paiement ?

**Q31.** Quelle limite supplémentaire cela met-il en évidence pour SMOTE après One-Hot Encoding ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

# 16. Choisir une stratégie

**Q32.** Parmi les stratégies expérimentées, laquelle retiendriez-vous ?

**Q33.** Quel seuil choisiriez-vous ?

**Q34.** Justifiez votre décision à partir des métriques observées et du coût métier des faux positifs et faux négatifs.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

# 17. Évaluation finale
Une fois la stratégie et le seuil choisis sur validation, utilisez le test **une seule fois**.

In [ ]:
# À VOUS DE JOUER

# 1. Regroupez train et validation.
# train_validation = ...

# 2. Reconstruisez X et y.
# ...

# 3. Appliquez sur train + validation la stratégie de rééchantillonnage retenue.
# ...

# 4. Réentraînez la régression logistique.
# ...

# 5. Calculez les probabilités sur test.
# ...

# 6. Appliquez le seuil choisi sur validation.
# ...

# 7. Calculez les métriques finales.
# ...

### Dernière analyse

**Q35.** Les performances test sont-elles proches de celles observées sur validation ?

**Q36.** Pourquoi ne faut-il pas modifier le seuil après avoir observé le test ?

**Q37.** Quelles décisions relevaient de l'apprentissage du modèle et lesquelles relevaient de la décision métier ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

# 18. Bilan

$$
\text{données}
\rightarrow
\text{probabilité}
\rightarrow
\text{seuil}
\rightarrow
\text{décision}
\rightarrow
\text{conséquences}
$$

À retenir :

- l'accuracy peut masquer l'échec sur la classe d'intérêt ;
- le seuil contrôle le compromis entre types d'erreurs ;
- précision et rappel répondent à des questions différentes ;
- ROC et Precision-Recall étudient le modèle au-delà d'un seuil unique ;
- le rééchantillonnage modifie ce que voit le modèle ;
- SMOTE peut aider mais aussi créer de l'ambiguïté lorsque les classes se chevauchent ;
- une observation synthétique n'est pas une observation réelle.

La question finale n'est donc pas seulement **« quel modèle prédit le mieux ? »**, mais aussi **« à partir de ces probabilités, quelle décision voulons-nous prendre ? »**